# Design Spaces 

## Setup

### Import Packages and Directories

In [1]:
import flowtangent as ft
import jax
import jax.numpy as jnp
import numpy as np

from pathlib import Path

from flowtangent.models.kernels import RBFKernel, Matern52Kernel
from flowtangent.models.surrogates import SVGPSurrogate
from flowtangent.models.manifolds import LinearPCA, Passthrough, ScalingManifold
from flowtangent.models.design import ActiveSpace, Parameter, Requirement

import jax.numpy as jnp

In [2]:
import string
from functools import lru_cache

@lru_cache(maxsize=2048)
def surf_lookup(name: str):
    safe_name = name.replace("-", "_").replace(".", "_")
    if safe_name and safe_name[0] in string.digits:
        safe_name = f"af_{safe_name}"

    af = getattr(ft.airfoils, safe_name)

    y_af = jnp.concatenate([af.y_upper, af.y_lower])
    return y_af

def name2surf(batch: dict):
    names = batch["foil_name"]
    
    # 1. Strip Xarray metadata if it slipped through
    if hasattr(names, "values"):
        names = names.values
        
    # 2. Force it into a flat, pure NumPy array
    names = np.asarray(names).flatten()
    
    # 3. Now iteration yields pure strings (or numpy.str_), which str() handles perfectly
    surfaces = [surf_lookup(str(name)) for name in names]
    
    batch["y_af"] = jnp.stack(surfaces)
    return batch

### Import Dataset

In [3]:
data_dir = Path("/home/jordan/dev/datasets/mUIUC_5D.zarr")
data_dir.mkdir(parents=True, exist_ok=True)

In [4]:
from huggingface_hub import hf_hub_download
import zipfile

zip_path = hf_hub_download(repo_id="eden-uiuc/mUIUC-5D", repo_type="dataset", filename="mUIUC_5D.zarr.zip")

if not (data_dir / 'zarr.json').exists():
    print("Extracting mUIUC 5D Dataset")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(data_dir)
    print("Extraction Complete.")
else:
    print ("Stored dataset detected.")

print(f"Dataset ready at: {data_dir}")

Stored dataset detected.
Dataset ready at: /home/jordan/dev/datasets/mUIUC_5D.zarr


In [4]:
XFD_raw = ft.utils.data.XArrayDataset(data_dir).to_tabular(["run", "alpha"])
XFD = XFD_raw.dropna(["alpha"])
# XFD = XFD.transform_variable(name2surf)
XFD._proxy._ds.load()

Dropping NaNs from dataset:   0%|          | 0/104 [00:00<?, ?it/s]

<xarray.Dataset> Size: 654MB
Dimensions:     (run: 358400, bend_var: 24, cond_var: 4, alpha: 42, polar_var: 9)
Coordinates:
  * bend_var    (bend_var) <U18 2kB 'Area' 'Slen' ... 'skin_Y_Ixx/t(Y-Yc)' 'J/t'
  * cond_var    (cond_var) <U10 160B 'flap_angle' 'flap_x/c' 'Re' 'Mach'
  * polar_var   (polar_var) <U7 252B 'alpha' 'CL' 'CD' ... 'Top_Itr' 'Bot_Itr'
Dimensions without coordinates: run, alpha
Data variables:
    bend_data   (run, bend_var) float32 34MB 0.0463 2.051 0.3786 ... 0.04929 nan
    conditions  (run, cond_var) float32 6MB -1.81 0.7673 ... 1.978e+05 0.5435
    foil_name   (run) <U50 72MB '2032c' '2032c' '2032c' ... 'ys930' 'ys930'
    polar_data  (run, alpha, polar_var) float32 542MB 0.0 0.5962 ... 5.937 86.99

In [5]:
unique_names = [str(n) for n in set(np.asarray(XFD_raw.foil_name))][1:]
surfs = [jnp.atleast_2d(surf_lookup(n)) for n in unique_names]
X_SURF = jnp.concatenate(surfs)

### Set up Parameters

In [6]:
# Inputs
y_af = Parameter(name="y_af", is_transform=True, transform_sources=("foil_name",))
Mach = Parameter(name="Mach")
Re = Parameter(name="Re")
flap_loc = Parameter(name="flap_x/c")
flap_angle = Parameter(name="flap_angle")
alpha = Parameter(name="alpha")

# Outputs
CL = Parameter(name="CL", is_output=True)
CD = Parameter(name="CD", is_output=True)
CM = Parameter(name="CM", is_output=True)

theta = (y_af, Mach, Re, flap_loc, flap_angle, alpha, CL, CD, CM)

### Set up Manifold

In [7]:
geofold = LinearPCA(physical_dim=256, latent_dim=16) # Airfoil Encoding
geofold = geofold.fit(X_SURF)

machfold    = ScalingManifold(method="softplus")
refold      = ScalingManifold(method="logarithmic")
hingefold   = ScalingManifold(method="algebraic_bounded", bounds_min=0.6, bounds_max=0.85)
deltafold   = ScalingManifold(method="linear", scale_factors=10)
alphafold   = ScalingManifold(method="linear", scale_factors=15)

mfold = geofold + machfold + refold + hingefold + deltafold + alphafold

### Setup Design Space

In [25]:
batch_size = 8192
beta = 0.5 * batch_size / len(XFD)

mu_scl =  (XFD.CL.mean().item(), XFD.CD.mean().item(), XFD.CM.mean().item())
sig_scl = (XFD.CL.std().item(), XFD.CD.std().item(), XFD.CM.std().item())

krnl = Matern52Kernel(input_dim=21)
surr = SVGPSurrogate(
    jax.random.key(42),
    input_dim=21,
    output_dim=3,
    kernel=krnl,
    num_inducing=1024,
    complexity_weight=beta,
    jitter=1e-5,
    scale_mean=mu_scl,
    scale_std=sig_scl
)

airspace = ActiveSpace(
    dataset=XFD,
    manifold=mfold,
    surrogate=surr,
    parameters=theta,
)

In [ ]:
airspace = airspace.fit_models(batch_size=batch_size, surrogate_epochs=5)

### Diagnostic Setup

In [10]:
def generate_diagnostics(model, test_x, test_y, n_bins=21):
    # 1. Base Predictions & Metrics
    preds = jax.vmap(model.surrogate.predict)(test_x)
    residuals = preds.means - test_y
    absolute_errors = jnp.abs(residuals)

    safe_y = jnp.where(jnp.abs(test_y) < 1e-4, 1e-4, jnp.abs(test_y))
    
    pred_std = jnp.sqrt(preds.variances)
    z_scores = residuals / pred_std

    metrics = {
        metr: {
            "MAE": jnp.mean(absolute_errors[:, i]).item(),
            "RMSE": jnp.sqrt(jnp.mean(residuals[:, i]**2)).item(),
            "Z_Score_Variance": jnp.var(z_scores[:, i]).item() 
        }
        for i, metr in enumerate(["CL", "CD", "CM"])
    }

    # 2. Vectorized Binning
    m_dist = jnp.linalg.norm(test_x[:, :16], axis=1)
    
    bins_dict = {
        "Geometry": jnp.linspace(0., jnp.max(m_dist), n_bins),
        "Mach": jnp.linspace(0.0, 0.6, n_bins),
        "Re": jnp.logspace(np.log10(50_000), np.log10(3_000_000), n_bins),
        "Pivot": jnp.linspace(0.6, 0.85, n_bins),
        "Angle": jnp.linspace(-10, 15, n_bins),
        "Alpha": jnp.linspace(-5, 15, n_bins)
    }

    test_x_hat = jax.vmap(model.reconstruct)(test_x)
    features_dict = {
        "Geometry": m_dist,
        "Mach": test_x_hat["Mach"].flatten(),
        "Re": test_x_hat["Re"].flatten(),
        "Pivot": test_x_hat["flap_x/c"].flatten(),
        "Angle": test_x_hat["flap_angle"].flatten(),
        "Alpha": test_x_hat["alpha"].flatten(),
    }
    
    param_names = list(bins_dict.keys())
    features = jnp.stack([features_dict[p] for p in param_names]) # (6_params, N_points)
    bins_stack = jnp.stack([bins_dict[p] for p in param_names])   # (6_params, n_bins)
    
    # Digitize all features at once. Indices 1 to (n_bins-1) correspond to our defined intervals.
    idxs = jax.vmap(lambda f, b: jnp.digitize(f, b, right=True))(features, bins_stack)
    
    # Create boolean mask: (6_params, N_points, n_bins + 1)
    n_indices = n_bins + 1
    masks = idxs[..., None] == jnp.arange(n_indices)
    
    # 3. Tensor Aggregation for Binned Means
    sq_frac_errs = ((residuals / safe_y) ** 2).T  # (3_metrics, N_points)
    
    # Sum errors across points in each bin: (3_metrics, 6_params, n_indices)
    sum_sq_frac = jnp.einsum('mn,pnb->mpb', sq_frac_errs, masks)
    
    # Count points in each bin: (6_params, n_indices)
    counts = jnp.sum(masks, axis=1)
    
    # Divide by count (MSE), then square root (RMSE)
    mean_sq_frac = jnp.where(counts > 0, sum_sq_frac / counts, np.nan)
    rms_frac_errors = jnp.sqrt(mean_sq_frac)
    
    valid_rms_frac = rms_frac_errors[:, :, 1:n_bins]
    
    errors = {}
    for p_idx, param in enumerate(param_names):
        b = bins_dict[param]
        centers = (b[:-1] + b[1:]) / 2.0
        
        errors[param] = {
            "bins": b,
            "centers": centers,
            "means": { 
                metr: valid_rms_frac[m_idx, p_idx, :]
                for m_idx, metr in enumerate(["CL", "CD", "CM"])
            }
        }
        
    return preds, metrics, z_scores, errors

In [11]:
from plotly import graph_objects as go
from plotly.subplots import make_subplots

def plot_error_matrices(errors):
    param_names = list(errors.keys())
    metrics = ["CL", "CD", "CM"]
    
    fig = make_subplots(
        rows=2, cols=3, 
        subplot_titles=param_names,
        horizontal_spacing=0.08,
        vertical_spacing=0.15
    )
    
    for i, param in enumerate(param_names):
        row = (i // 3) + 1
        col = (i % 3) + 1
        
        centers = errors[param]["centers"]
        z_data = [errors[param]["means"][m] for m in metrics]
        
        fig.add_trace(
            go.Heatmap(
                z=z_data,
                x=centers,
                y=metrics,
                colorscale="Viridis", 
                zmin=0.0, 
                zmax=2.0,
                # Optional: set zmax=0.5 (50%) to prevent zero-crossing spikes from washing out the color scale
                showscale=(i == 0),
                colorbar=dict(
                    title="RMS % Error",
                    tickformat=".1%"  # Automatically formats 0.15 as 15.0%
                ) if i == 0 else None,
            ),
            row=row, col=col
        )
        
        if param == "Re":
            fig.update_xaxes(type="log", row=row, col=col)
            
    fig.update_layout(
        title_text="Surrogate RMS Percentage Error",
        height=600, 
        width=1000,
        plot_bgcolor='white'
    )
    
    return fig

### Diagnostics

In [ ]:
# 1. Generate globally random indices
total_points = len(airspace.dataset)
test_indices = np.random.choice(total_points, size=8192, replace=False)

# Sort indices to ensure Zarr reads the chunks sequentially without thrashing
test_indices.sort()

# 2. Extract the batch instantly via the new vectorized fast-path
raw_batch = airspace.dataset[test_indices]

geo_x = raw_batch["y_af"]

# 3. Stack the inputs and targets
scalar_x = jnp.stack([
    raw_batch["Mach"], 
    raw_batch["Re"], 
    raw_batch["flap_x/c"], 
    raw_batch["flap_angle"], 
    raw_batch["alpha"]
], axis=-1)

raw_x = jnp.concatenate([geo_x, scalar_x], axis=-1)

test_y = jnp.stack([
    raw_batch["CL"], 
    raw_batch["CD"], 
    raw_batch["CM"]
], axis=-1)

# 4. Push through the manifold
test_x = jax.vmap(airspace.manifold.encode)(raw_x)

# 5. Generate matrices
preds, metrics, z_scores, errors = generate_diagnostics(airspace, test_x, test_y)


In [30]:
plot_error_matrices(errors)

### Hyperparameter Search

In [ ]:
from collections import deque

from ray import train, tune
from ray.tune.schedulers import ASHAScheduler

import equinox as eqx

batch_size = 8192
beta = 0.5 * batch_size / len(XFD)

mu_scl =  (XFD.CL.mean().item(), XFD.CD.mean().item(), XFD.CM.mean().item())
sig_scl = (XFD.CL.std().item(), XFD.CD.std().item(), XFD.CM.std().item())

def train_svgp_trial(config, XFD_data, mfold_obj, theta_params, mu_scl, sig_scl):
    # 1. Unpack hyperparams
    num_inducing = config["num_inducing"]
    jitter = config["jitter"]
    complexity_weight = config["complexity_weight"]
    use_scaling = config["use_scaling"]
    
    # 2. Setup Data (Extract global scale factors if scaling is enabled)
    scale_mean = mu_scl if use_scaling else None
    scale_std = sig_scl if use_scaling else None

    @lru_cache(maxsize=2048)
    def local_surf_lookup(name: str):
        safe_name = name.replace("-", "_").replace(".", "_")
        if safe_name and safe_name[0] in string.digits:
            safe_name = f"af_{safe_name}"

        af = getattr(ft.airfoils, safe_name)

        y_af = jnp.concatenate([af.y_upper, af.y_lower])
        return y_af

    XFD_data = XFD_data.transform_variable(local_surf_lookup)
    
    # 3. Initialize Model
    surr = SVGPSurrogate(
        jax.random.key(42),
        kernel=Matern52Kernel(21),
        input_dim=21, 
        output_dim=3,
        num_inducing=num_inducing,
        jitter=jitter,
        complexity_weight=complexity_weight,
        scale_mean=scale_mean,
        scale_std=scale_std
    )

    airspace = ActiveSpace(
        dataset=XFD_data,
        manifold=mfold_obj,
        surrogate=surr,
        parameters=theta_params,
    )

    recent_nlpds = deque(maxlen=50)

    def NLPD_Plateau(metrics):
        # Report to Ray so ASHA can rank and kill trials
        train.report(metrics)
        
        # Implement manual plateau stopping
        nlpd = metrics.get("NLPD", 0.0)
        recent_nlpds.append(nlpd)
        
        # Act as grace_period=100 and std threshold
        # (Assuming 'metrics' includes the current epoch number)
        if metrics.get("epoch", 0) >= 100 and len(recent_nlpds) == 50:
            if np.std(recent_nlpds) < 0.01:
                return True  # Tells train_model to break early
                
        return False  # Continue training

    ZDL = ft.utils.data.LatentDataLoader(airspace.get_dataloader(batch_size=batch_size), mfold_obj)
    airspace.surrogate.fit(ZDL, epochs=config["max_epochs"], stop_fn=NLPD_Plateau)

In [ ]:
from dataclasses import replace

def run_hyperparameter_sweep():
    # Define the search space
    search_space = {
        "num_inducing": tune.choice([128, 256, 512, 1024, 2048]),
        "jitter": tune.loguniform(1e-6, 1e-2),
        "complexity_weight": tune.loguniform(1e-4, 1.0),
        "use_scaling": tune.choice([True, False]),
        "max_epochs": 2000  # Set high; ASHA and EarlyStopping will cut it short
    }

    # ASHA Scheduler: Halves the number of running trials at each grace period
    scheduler = ASHAScheduler(
        time_attr="training_iteration",
        metric="NLPD",
        mode="min",
        max_t=2000,
        grace_period=50,       # Let every trial run at least 50 epochs before judging
        reduction_factor=2     # Kill the bottom 50% of trials at each milestone
    )
    
    trainable_with_params = tune.with_parameters(
        train_svgp_trial,
        XFD_data=XFD,
        mfold_obj=mfold,
        theta_params=theta,
        mu_scl=mu_scl,
        sig_scl=sig_scl
    )

    tuner = tune.Tuner(
        tune.with_resources(trainable_with_params, resources={"cpu": 12, "gpu": 1.0}),
        param_space=search_space,
        tune_config=tune.TuneConfig(
            num_samples=8,      
            scheduler=scheduler,
        ),
        run_config=tune.RunConfig(
            name="svgp_baseline_sweep",
            checkpoint_config=tune.CheckpointConfig(
                checkpoint_score_attribute="NLPD",
                checkpoint_score_order="min",
                num_to_keep=3
            )
        )
    )

    results = tuner.fit()
    
    # Extract the best configuration
    best_result = results.get_best_result("NLPD", "min")
    print("Best hyperparameters found:", best_result.config)

In [23]:
run_hyperparameter_sweep()

(train_svgp_trial pid=3301117) Can't get attribute 'surf_lookup' on <module '__main__' from '/home/jordan/dev/EDEn/flowtangent/.venv/lib/python3.12/site-packages/ray/_private/workers/default_worker.py'>
(train_svgp_trial pid=3301117) Traceback (most recent call last):
(train_svgp_trial pid=3301117)   File "/home/jordan/dev/EDEn/flowtangent/.venv/lib/python3.12/site-packages/ray/_private/serialization.py", line 595, in deserialize_objects
(train_svgp_trial pid=3301117)     obj = self._deserialize_object(
(train_svgp_trial pid=3301117)           ^^^^^^^^^^^^^^^^^^^^^^^^^
(train_svgp_trial pid=3301117)   File "/home/jordan/dev/EDEn/flowtangent/.venv/lib/python3.12/site-packages/ray/_private/serialization.py", line 437, in _deserialize_object
(train_svgp_trial pid=3301117)     return self._deserialize_msgpack_data(
(train_svgp_trial pid=3301117)            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(train_svgp_trial pid=3301117)   File "/home/jordan/dev/EDEn/flowtangent/.venv/lib/python3.12/site-pack

RuntimeError: No best trial found for the given metric: NLPD. This means that no trial has reported this metric, or all values reported for this metric are NaN. To not ignore NaN values, you can set the `filter_nan_and_inf` arg to False.